# Tag Preprocessing
We read the `data/tags/tag_list.csv` and perform some pre-processing steps to obtain a matching file.


In [1]:
import pandas as pd

df = pd.read_csv("data/tags/tag_list.csv")


In [2]:
import re
import json
from collections import defaultdict
import yaml

def _split_additional(val):
    if pd.isna(val):
        return []
    if isinstance(val, (list, tuple, set)):
        return [str(v).strip() for v in val if str(v).strip()]
    s = str(val)
    parts = re.split(r'[;,|/]+', s)
    return [p.strip() for p in parts if p.strip()]

# build initial map: tag -> set(cues) (normalized to lowercase)
cues_map = {}
for tag, grp in df.groupby('tag'):
    cues = set()
    cues.add(str(tag))
    for v in grp['tag_alternative'].dropna().unique():
        cues.update(_split_additional(v))
    # normalize cues to lowercase for matching
    cues_map[tag] = {c.strip().lower() for c in cues if c and str(c).strip()}

# union-find to merge tags with overlapping cues
parents = {t: t for t in cues_map}

def find(x):
    while parents[x] != x:
        parents[x] = parents[parents[x]]
        x = parents[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parents[rb] = ra

tags = list(cues_map.keys())
for i in range(len(tags)):
    for j in range(i + 1, len(tags)):
        if cues_map[tags[i]] & cues_map[tags[j]]:  # overlap -> merge
            union(tags[i], tags[j])

# aggregate merged groups
groups = defaultdict(set)
group_cues = defaultdict(set)
for t in tags:
    r = find(t)
    groups[r].add(t)
    group_cues[r].update(cues_map[t])

# produce final tag_dict: key = representative tag (lexicographically smallest), value = sorted list of cues
tag_dict = {}
for root, tagset in groups.items():
    key = min(tagset)
    tag_dict[key] = sorted(group_cues[root])

tag_dict = {}
for root, tagset in groups.items():
    cues_for_group = group_cues[root]
    nonspace_cues = [c for c in cues_for_group if ' ' not in c]
    if nonspace_cues:
        key = min(nonspace_cues)
    else:
        nonspace_tags = [t for t in tagset if ' ' not in t]
        key = min(nonspace_tags) if nonspace_tags else min(tagset)
    tag_dict[key] = sorted(cues_for_group)

with open("data/tags/tags.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(tag_dict, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

tag_dict


{'60s': ['60s', 'sixties'],
 '70s': ['70s', 'seventies'],
 '80s': ['80s', 'eighties'],
 '90s': ['90s', 'ninties'],
 'accordion': ['accordion'],
 'acidjazz': ['acid jazz', 'acidjazz'],
 'acoustic': ['acoustic'],
 'acousticguitar': ['acoustic guitar', 'acousticguitar'],
 'acousticbassguitar': ['acoustic bass guitar', 'acousticbassguitar'],
 'action': ['action'],
 'adventure': ['adventure'],
 'advertising': ['advertising'],
 'african': ['african'],
 'airy': ['airy'],
 'alternative': ['alternative'],
 'alternativerock': ['alternative rock', 'alternativerock'],
 'ambient': ['ambient'],
 'ambiental': ['ambiental'],
 'arabic': ['arabic'],
 'atmospheric': ['atmospheric'],
 'background': ['background'],
 'backing track': ['backing track'],
 'ballad': ['ballad'],
 'banjo': ['banjo'],
 'baroque': ['baroque'],
 'bass': ['bass'],
 'beat': ['beat'],
 'beats': ['beats'],
 'bell': ['bell'],
 'bells': ['bells'],
 'birds': ['birds'],
 'blues': ['blues'],
 'bluesrock': ['bluesrock'],
 'bongo': ['bongo'],

## Prompting an LLM to translate the dict
```You are an expert in the music domain. Create a multilingual version of the dictionary of tags below. The structure in the resulting dict should be the following: The keys are the same as the ones in the initial dictionary. However, the values should be subdicts with keys per language (English, Spanish, French, Portuguese, German, Italian) mapping to the respective translations. Please always consider plurals and conjugations if applicable. Furthermore, please only consider translations which match the context of the music domain.``` 

In [1]:
import yaml

with open("data/tags/tags_multilingual.yaml", "r", encoding="utf-8") as f:
    tags_ml = yaml.safe_load(f)

# keeping the first occurrence (iteration order of the subdict)
for tag, subdict in tags_ml.items():
    seen = set()
    for lang in list(subdict.keys()):
        lst = subdict.get(lang) or []
        cleaned = []
        for val in lst:
            if val not in seen:
                cleaned.append(val)
                seen.add(val)
        subdict[lang] = cleaned

with open("data/tags/tags_multilingual.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(tags_ml, f, allow_unicode=True, sort_keys=True, default_flow_style=False)
    